# Video Data Augmentation and Classification

This notebook builds a tiny **synthetic video classification** task:
- Class 0: square moves left to right
- Class 1: square moves top to bottom

A/B test:
- baseline training
- augmented training with noise and horizontal flip

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class MovingSquareVideoDataset(Dataset):
    def __init__(self, n_samples=1200, frames=8, size=32, augment=False):
        self.n_samples = n_samples
        self.frames = frames
        self.size = size
        self.augment = augment

    def __len__(self):
        return self.n_samples

    def _make_clip(self, label):
        clip = np.zeros((self.frames, self.size, self.size), dtype=np.float32)
        for t in range(self.frames):
            frame = np.zeros((self.size, self.size), dtype=np.float32)
            if label == 0:
                x = 2 + t * 2
                y = 10
            else:
                x = 10
                y = 2 + t * 2
            frame[y:y+6, x:x+6] = 1.0
            clip[t] = frame
        return clip

    def __getitem__(self, idx):
        label = idx % 2
        clip = self._make_clip(label)

        if self.augment:
            if np.random.rand() < 0.5:
                clip = clip[:, :, ::-1].copy()
            clip = clip + np.random.normal(0, 0.05, clip.shape).astype(np.float32)
            clip = np.clip(clip, 0, 1)

        clip = torch.tensor(clip).unsqueeze(0)  # (C, T, H, W)
        label = torch.tensor(label, dtype=torch.long)
        return clip, label

base_ds = MovingSquareVideoDataset(augment=False)
aug_ds = MovingSquareVideoDataset(augment=True)
test_ds = MovingSquareVideoDataset(n_samples=400, augment=False)

base_loader = DataLoader(base_ds, batch_size=32, shuffle=True)
aug_loader = DataLoader(aug_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

In [ ]:
class Small3DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),
            nn.Conv3d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),
            nn.Flatten(),
            nn.Linear(16 * 2 * 8 * 8, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )
    def forward(self, x):
        return self.net(x)

def train_and_eval(train_loader, epochs=5):
    model = Small3DCNN().to(device)
    crit = nn.CrossEntropyLoss()
    opt = optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

base_acc = train_and_eval(base_loader)
aug_acc = train_and_eval(aug_loader)

print("Baseline accuracy:", round(base_acc, 4))
print("Augmented accuracy:", round(aug_acc, 4))